In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:

config_path = "config/binaural_attn/word_task_v10_backbone_word_config.yaml"
# config_path = "config/binaural_attn/word_task_half_co_loc_v06.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 192
config['audio']['rep_kwargs']['rep_on_gpu'] = True
# print(f"Default lr is {config['hparas']['lr']}")
# config['hparas']['lr'] = 0.000001
# print(f"Trying lr = {config['hparas']['lr']}")



In [3]:
seed_everything(0)
importlib.reload(binaural_lightning)

module = binaural_lightning.BinauralAttentionModule(config)

[rank: 0] Seed set to 0


Using explicit dim specification for demeaning in audio transforms
Using BackBoneCNN
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv -> ReLU
Using dataset BinauralWordRecDataset
OptimizedModule(
  (_orig_mod): BackBoneCNN(
    (model_dict): ModuleDict(
      (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
      (conv_block_0): Sequential(
        (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
        (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
        (2): ReLU()
      )
      (hann_pool_0): HannPooling2d()
      (conv_block_1): Sequential(
        (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
        (1): Conv2d(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
        (2): ReLU()
      )
      (hann_pool_1): HannPooling2d()
      (conv_block_2): Sequential(
        (0): LayerNorm((64, 10, 1245), eps=1e-05, elementwise_affine=True)
        (1): Co

/orcd/data/jhm/001/imgriff/projects/Auditory-Attention/src/audio_transforms.py:335: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  self.downsampling_op = self.downsampling(self.sr,


In [ ]:
trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    gradient_clip_val=10,
    gradient_clip_algorithm="value",
    accelerator="gpu",
)
trainer.fit(module)

/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/ ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them

/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
735 files in train concat dataset
len training set = 1195074
Epoch 0:   0%|          | 12/6225 [02:51<24:37:13,  0.07it/s, v_num=3.86e+7, train_loss=6.690, grad_norm=1.440]